<a href="https://colab.research.google.com/github/JessieMorozov/Exercise_Form_Detection/blob/main/squat_form_mvp_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Installs

Run this cell in Colab. For local use, install `requirements.txt` instead

In [ ]:
try:
    import google.colab

    !apt-get -y install ffmpeg
    !pip -q install yt-dlp mediapipe opencv-python pandas numpy matplotlib scikit-learn joblib

except ImportError:
    print("Running outside Colab. Install dependencies from requirements.txt")

## 2. Imports and paths

In [ ]:
import json
import os
import re
import shutil
import subprocess
from pathlib import Path

import cv2
import joblib
import mediapipe as mp
import numpy as np
import pandas as pd

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.base import clone
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedShuffleSplit,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


try:
    from google.colab import drive

    drive.mount("/content/drive")

    DEFAULT_ROOT = "/content/drive/MyDrive/YOUR_PROJECT_FOLDER"

except ImportError:
    DEFAULT_ROOT = "./physio_project"

PROJECT_ROOT = Path(
    os.getenv("PHYSIO_PROJECT_ROOT", DEFAULT_ROOT)
)

DATASET_ROOT = PROJECT_ROOT / "dataset"

METADATA_DIR = DATASET_ROOT / "metadata"
CLIPS_DIR = DATASET_ROOT / "clips"

EXPORTS_DIR = DATASET_ROOT / "exports"

ARTIFACT_DIR = EXPORTS_DIR / "features"
MODEL_DIR = EXPORTS_DIR / "trained_models"
VECTORS_ZIP_DIR = EXPORTS_DIR / "vectors_zip"

MASTER_INDEX_PATH = METADATA_DIR / "master_index.csv"

for path in [
    METADATA_DIR,
    CLIPS_DIR,
    ARTIFACT_DIR,
    MODEL_DIR,
    VECTORS_ZIP_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
MODEL_PATH = PROJECT_ROOT / "models" / "pose_landmarker_lite.task"

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

## 3. Load and audit clip registry

In [ ]:
def load_master_index(path=MASTER_INDEX_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing master index: {path}")
    return pd.read_csv(path)


def is_truthy_usable(series):

    return series.astype(str).str.strip().str.lower().isin(["1", "true", "yes", "y"])


def dataset_summary(df=None):
    if df is None:
        df = load_master_index()

    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total clips: {len(df)}")

    for col in ["view_profile", "good_form", "exercise_type", "load_type", "label_confidence", "view_confidence", "usable"]:
        if col in df.columns:
            print(f"
{col.upper()} COUNTS")
            print("-" * 40)
            print(df[col].fillna("MISSING").value_counts().sort_index())

    if {"view_profile", "good_form"}.issubset(df.columns):
        print("
VIEW x LABEL TABLE")
        print("-" * 40)
        display(pd.crosstab(df["view_profile"], df["good_form"], margins=True))

    return df

master_df = dataset_summary()
display(master_df.head())

## 4. Timestamp parsing

In [ ]:
def parse_timestamp_to_seconds(ts):

    if pd.isna(ts):
        raise ValueError("Timestamp is missing.")
    if isinstance(ts, (int, np.integer)):
        return int(ts)
    if isinstance(ts, (float, np.floating)) and float(ts).is_integer():
        return int(ts)

    s = str(ts).strip()
    if s == "":
        raise ValueError("Timestamp is blank.")
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return int(float(s))

    parts = s.split(":")
    if not all(re.fullmatch(r"\d+(\.\d+)?", p.strip()) for p in parts):
        raise ValueError(f"Invalid timestamp format: {ts}")

    nums = [float(p) for p in parts]
    if len(nums) == 2:
        minutes, seconds = nums
        total = minutes * 60 + seconds
    elif len(nums) == 3:
        hours, minutes, seconds = nums
        total = hours * 3600 + minutes * 60 + seconds
    else:
        raise ValueError(f"Invalid timestamp format: {ts}")

    return int(round(total))


def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    rem = seconds % 3600
    m = rem // 60
    s = rem % 60
    return f"{h}:{m:02d}:{s:02d}" if h > 0 else f"{m}:{s:02d}"


def standardize_master_timestamps(save=True):
    df = load_master_index().copy()
    df["start_sec"] = df["start_time"].apply(parse_timestamp_to_seconds)
    df["end_sec"] = df["end_time"].apply(parse_timestamp_to_seconds)

    bad = df[df["end_sec"] <= df["start_sec"]]
    if len(bad) > 0:
        display(bad[["clip_id", "url", "start_time", "end_time", "start_sec", "end_sec"]])
        raise ValueError("Some clips have end_sec <= start_sec.")

    if save:
        df.to_csv(MASTER_INDEX_PATH, index=False)
        print("Saved standardized master_index.csv with start_sec/end_sec columns.")
    return df

master_df = standardize_master_timestamps(save=True)
display(master_df[["clip_id", "start_time", "end_time", "start_sec", "end_sec", "view_profile", "good_form"]].head())

## 5. Video download and frame extraction

In [ ]:
def run_command(cmd, verbose=True):
    if verbose:
        print("Running:", " ".join(str(x) for x in cmd))
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if verbose:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")
    return result.stdout


def download_clip_segment(url, start_sec, end_sec, out_path, max_height=720, overwrite=False):
    out_path = Path(out_path)
    if out_path.exists() and not overwrite:
        print(f"Raw video already exists, skipping download: {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)
    section = f"*{int(start_sec)}-{int(end_sec)}"
    cmd = [
        "yt-dlp",
        "-f", f"bv*[height<={max_height}]+ba/b[height<={max_height}]/best[height<={max_height}]/best",
        "--merge-output-format", "mp4",
        "--download-sections", section,
        "--force-keyframes-at-cuts",
        "-o", str(out_path),
        url,
    ]
    run_command(cmd, verbose=True)

    if not out_path.exists():
        candidates = sorted(out_path.parent.glob(out_path.stem + "*"))
        if candidates:
            candidates[0].rename(out_path)
    if not out_path.exists():
        raise FileNotFoundError(f"Download did not produce expected file: {out_path}")
    return out_path


def clear_directory(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def extract_frames_from_video(video_path, frames_dir, sample_fps=10, overwrite=False, image_ext="jpg"):
    video_path = Path(video_path)
    frames_dir = Path(frames_dir)

    if frames_dir.exists() and any(frames_dir.glob(f"*.{image_ext}")) and not overwrite:
        existing = sorted(frames_dir.glob(f"*.{image_ext}"))
        print(f"Frames already exist, skipping extraction: {len(existing)} frames")
        return existing

    clear_directory(frames_dir)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    source_fps = cap.get(cv2.CAP_PROP_FPS)
    if source_fps is None or source_fps <= 0 or np.isnan(source_fps):
        source_fps = 30.0
    stride = max(1, int(round(source_fps / sample_fps)))

    frame_idx, saved_idx, saved_paths = 0, 0, []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % stride == 0:
            out_path = frames_dir / f"frame_{saved_idx:05d}.{image_ext}"
            cv2.imwrite(str(out_path), frame)
            saved_paths.append(out_path)
            saved_idx += 1
        frame_idx += 1

    cap.release()
    print(f"Extracted {len(saved_paths)} frames from {video_path.name}")
    return saved_paths

## 6. MediaPipe pose extraction

In [ ]:
MODEL_PATH = PROJECT_ROOT / "models" / "pose_landmarker_lite.task"

def ensure_mediapipe_pose_model():
    if MODEL_PATH.exists():
        return MODEL_PATH

    raise FileNotFoundError(
        f"Missing MediaPipe model: {MODEL_PATH}\n"
        "Download pose_landmarker_lite.task manually and place it in PROJECT_ROOT/models/"
    )


MEDIAPIPE_IDXS = {
    "nose": 0,
    "left_shoulder": 11, "right_shoulder": 12,
    "left_hip": 23, "right_hip": 24,
    "left_knee": 25, "right_knee": 26,
    "left_ankle": 27, "right_ankle": 28,
    "left_foot_index": 31, "right_foot_index": 32,
}


def rotate_points(points, angle_rad):
    R = np.array([[np.cos(angle_rad), -np.sin(angle_rad)], [np.sin(angle_rad), np.cos(angle_rad)]], dtype=np.float32)
    return points @ R.T


def normalize_pose_2d(points, left_hip_idx=23, right_hip_idx=24, left_shoulder_idx=11, right_shoulder_idx=12):
    """Center at hips, rotate hips horizontal, and scale by torso length.

    This improves body-size comparability but may change the meaning of some camera-view-dependent features.
    """
    points = np.asarray(points, dtype=np.float32).copy()
    left_hip, right_hip = points[left_hip_idx], points[right_hip_idx]
    hip_center = (left_hip + right_hip) / 2.0
    points -= hip_center

    hip_vector = right_hip - left_hip
    points = rotate_points(points, -np.arctan2(hip_vector[1], hip_vector[0]))

    shoulder_center = (points[left_shoulder_idx] + points[right_shoulder_idx]) / 2.0
    torso_length = np.linalg.norm(shoulder_center)
    if torso_length <= 1e-6:
        hip_width = np.linalg.norm(points[right_hip_idx] - points[left_hip_idx])
        torso_length = hip_width if hip_width > 1e-6 else 1.0

    return points / torso_length


def landmark_vector(points):
    pts = np.asarray(points, dtype=np.float32)
    if pts.shape != (33, 2):
        raise ValueError(f"Expected pose shape (33, 2), got {pts.shape}")
    return pts.reshape(-1)


def get_pose_landmarks_from_image(image_bgr, landmarker):
    h, w = image_bgr.shape[:2]
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    detection_result = landmarker.detect(mp_image)

    if not detection_result.pose_landmarks:
        return None, None

    pose_landmarks = detection_result.pose_landmarks[0]
    points_px, visibility = [], []
    for lm in pose_landmarks:
        points_px.append([lm.x * w, lm.y * h])
        visibility.append(getattr(lm, "visibility", np.nan))

    return np.array(points_px, dtype=np.float32), np.array(visibility, dtype=np.float32)


def extract_pose_vectors_from_frames(frames_dir, out_csv_path, model_path=MODEL_PATH, overwrite=False):
    frames_dir = Path(frames_dir)
    out_csv_path = Path(out_csv_path)

    if out_csv_path.exists() and not overwrite:
        print(f"Vector CSV already exists, skipping: {out_csv_path}")
        return pd.read_csv(out_csv_path)

    frame_paths = sorted([p for p in frames_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    if not frame_paths:
        raise ValueError(f"No frames found in {frames_dir}")

    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )

    rows = []
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        for frame_idx, frame_path in enumerate(frame_paths):
            image_bgr = cv2.imread(str(frame_path))
            if image_bgr is None:
                continue
            points_px, visibility = get_pose_landmarks_from_image(image_bgr, landmarker)
            if points_px is None:
                continue
            points_norm = normalize_pose_2d(points_px)
            vec = landmark_vector(points_norm)
            row = {"frame_name": frame_path.name, "frame_idx": frame_idx}
            row.update({f"v{i}": float(val) for i, val in enumerate(vec)})
            row.update({f"vis{j}": float(vis) if not np.isnan(vis) else np.nan for j, vis in enumerate(visibility)})
            rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise ValueError(f"MediaPipe found no usable poses in {frames_dir}")

    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv_path, index=False)
    print(f"Saved {len(df)} pose-vector rows to {out_csv_path}")
    return df

## 7. Process clips from registry

In [ ]:
def process_one_clip_from_registry(clip_id, sample_fps=10, keep_raw_video=False, keep_frames=True, overwrite=False):
    df = load_master_index()
    row_df = df[df["clip_id"] == clip_id]
    if len(row_df) != 1:
        raise ValueError(f"Expected exactly one row for {clip_id}, found {len(row_df)}")

    row = row_df.iloc[0].to_dict()
    clip_dir = CLIPS_DIR / clip_id
    frames_dir = clip_dir / "frames"
    raw_video_path = clip_dir / "raw_video.mp4"
    vector_csv_path = clip_dir / "pose_vectors.csv"
    clip_dir.mkdir(parents=True, exist_ok=True)

    start_sec = int(row["start_sec"]) if "start_sec" in row and not pd.isna(row["start_sec"]) else parse_timestamp_to_seconds(row["start_time"])
    end_sec = int(row["end_sec"]) if "end_sec" in row and not pd.isna(row["end_sec"]) else parse_timestamp_to_seconds(row["end_time"])

    ensure_mediapipe_pose_model()
    download_clip_segment(row["url"], start_sec, end_sec, raw_video_path, overwrite=overwrite)
    extract_frames_from_video(raw_video_path, frames_dir, sample_fps=sample_fps, overwrite=overwrite)
    vec_df = extract_pose_vectors_from_frames(frames_dir, vector_csv_path, overwrite=overwrite)

    if not keep_raw_video and raw_video_path.exists():
        raw_video_path.unlink()
    if not keep_frames and frames_dir.exists():
        shutil.rmtree(frames_dir)

    return vec_df


def process_all_registry_clips(only_usable=True, limit=None, sample_fps=10, keep_raw_video=False, keep_frames=True, overwrite=False):
    df = load_master_index().copy()
    if only_usable and "usable" in df.columns:
        df = df[is_truthy_usable(df["usable"])]
    if limit is not None:
        df = df.head(limit)

    status_rows = []
    for _, row in df.iterrows():
        clip_id = row["clip_id"]
        print("
" + "=" * 80)
        print(f"Processing {clip_id}")
        print("=" * 80)
        try:
            vec_df = process_one_clip_from_registry(
                clip_id=clip_id,
                sample_fps=sample_fps,
                keep_raw_video=keep_raw_video,
                keep_frames=keep_frames,
                overwrite=overwrite,
            )
            status_rows.append({"clip_id": clip_id, "status": "ok", "num_vector_frames": len(vec_df), "error": ""})
        except Exception as exc:
            print(f"FAILED {clip_id}: {exc}")
            status_rows.append({"clip_id": clip_id, "status": "failed", "num_vector_frames": 0, "error": str(exc)})

    status_df = pd.DataFrame(status_rows)
    status_df.to_csv(METADATA_DIR / "processing_status.csv", index=False)
    display(status_df)
    return status_df

# mini test:
# status_df = process_all_registry_clips(limit=3, sample_fps=10)

# full run:
# status_df = process_all_registry_clips(sample_fps=10, keep_raw_video=False, keep_frames=True)

## 8. Feature engineering

In [ ]:
def load_vector_csv_as_points(csv_path):
    df = pd.read_csv(csv_path)
    coord_cols = [f"v{i}" for i in range(66)]
    missing = [c for c in coord_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing vector columns in {csv_path}: {missing[:5]}...")

    points_seq = []
    for _, row in df.iterrows():
        vec = row[coord_cols].to_numpy(dtype=np.float32)
        if not np.isnan(vec).any():
            points_seq.append(vec.reshape(33, 2))
    if len(points_seq) == 0:
        raise ValueError(f"No usable vector rows in {csv_path}")
    return points_seq, df


def angle_between_three_points(a, b, c):
    a, b, c = np.asarray(a, dtype=np.float32), np.asarray(b, dtype=np.float32), np.asarray(c, dtype=np.float32)
    ba, bc = a - b, c - b
    norm_ba, norm_bc = np.linalg.norm(ba), np.linalg.norm(bc)
    if norm_ba < 1e-6 or norm_bc < 1e-6:
        return np.nan
    cos_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))


def euclidean_distance(p1, p2):
    return float(np.linalg.norm(np.asarray(p2) - np.asarray(p1)))


def safe_divide(a, b, default=np.nan):
    if b is None or abs(b) < 1e-8:
        return default
    return float(a / b)


def torso_lean_deg(shoulder_center, hip_center):
    vec = np.asarray(shoulder_center) - np.asarray(hip_center)
    vertical = np.array([0.0, -1.0], dtype=np.float32)
    norm_vec = np.linalg.norm(vec)
    if norm_vec < 1e-6:
        return np.nan
    cos_angle = np.dot(vec, vertical) / norm_vec
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))


def extract_frame_features(p, idxs=MEDIAPIPE_IDXS):
    p = np.asarray(p, dtype=np.float32)
    l_shoulder, r_shoulder = p[idxs["left_shoulder"]], p[idxs["right_shoulder"]]
    l_hip, r_hip = p[idxs["left_hip"]], p[idxs["right_hip"]]
    l_knee, r_knee = p[idxs["left_knee"]], p[idxs["right_knee"]]
    l_ankle, r_ankle = p[idxs["left_ankle"]], p[idxs["right_ankle"]]
    l_foot, r_foot = p[idxs["left_foot_index"]], p[idxs["right_foot_index"]]

    shoulder_center = (l_shoulder + r_shoulder) / 2.0
    hip_center = (l_hip + r_hip) / 2.0
    knee_center = (l_knee + r_knee) / 2.0
    ankle_center = (l_ankle + r_ankle) / 2.0

    left_knee_angle = angle_between_three_points(l_hip, l_knee, l_ankle)
    right_knee_angle = angle_between_three_points(r_hip, r_knee, r_ankle)
    left_hip_angle = angle_between_three_points(l_shoulder, l_hip, l_knee)
    right_hip_angle = angle_between_three_points(r_shoulder, r_hip, r_knee)

    left_femur = euclidean_distance(l_hip, l_knee)
    right_femur = euclidean_distance(r_hip, r_knee)
    left_tibia = euclidean_distance(l_knee, l_ankle)
    right_tibia = euclidean_distance(r_knee, r_ankle)
    torso_len = euclidean_distance(shoulder_center, hip_center)
    mean_femur = np.nanmean([left_femur, right_femur])
    mean_tibia = np.nanmean([left_tibia, right_tibia])

    hip_width = abs(r_hip[0] - l_hip[0])
    knee_width = abs(r_knee[0] - l_knee[0])
    ankle_width = abs(r_ankle[0] - l_ankle[0])
    knee_ankle_ratio = safe_divide(knee_width, ankle_width)
    valgus_score_global = 1.0 - knee_ankle_ratio if not np.isnan(knee_ankle_ratio) else np.nan

    left_knee_inward = max(0.0, l_knee[0] - l_ankle[0])
    right_knee_inward = max(0.0, r_ankle[0] - r_knee[0])

    return {
        "left_knee_angle_deg": left_knee_angle,
        "right_knee_angle_deg": right_knee_angle,
        "mean_knee_angle_deg": np.nanmean([left_knee_angle, right_knee_angle]),
        "knee_angle_asym_deg": abs(left_knee_angle - right_knee_angle),
        "left_hip_angle_deg": left_hip_angle,
        "right_hip_angle_deg": right_hip_angle,
        "mean_hip_angle_deg": np.nanmean([left_hip_angle, right_hip_angle]),
        "hip_angle_asym_deg": abs(left_hip_angle - right_hip_angle),
        "torso_lean_deg": torso_lean_deg(shoulder_center, hip_center),
        "femur_tibia_ratio": safe_divide(mean_femur, mean_tibia),
        "torso_femur_ratio": safe_divide(torso_len, mean_femur),
        "hip_to_ankle_x_norm_femur": safe_divide(abs(hip_center[0] - ankle_center[0]), mean_femur),
        "knee_to_ankle_x_norm_tibia": safe_divide(abs(knee_center[0] - ankle_center[0]), mean_tibia),
        "hip_width": hip_width,
        "knee_width": knee_width,
        "ankle_width": ankle_width,
        "knee_ankle_ratio": knee_ankle_ratio,
        "valgus_score_global": valgus_score_global,
        "left_knee_inward_norm": safe_divide(left_knee_inward, ankle_width),
        "right_knee_inward_norm": safe_divide(right_knee_inward, ankle_width),
        "valgus_asymmetry": abs(safe_divide(left_knee_inward, ankle_width, 0.0) - safe_divide(right_knee_inward, ankle_width, 0.0)),
        "stance_width_norm_torso": safe_divide(ankle_width, torso_len),
        "foot_width_norm_torso": safe_divide(abs(r_foot[0] - l_foot[0]), torso_len),
    }

In [ ]:
def summarize_series(series, prefix):
    arr = np.asarray(series, dtype=np.float32)
    arr = arr[~np.isnan(arr)]
    stats = ["mean", "std", "min", "max", "range", "q25", "median", "q75", "slope"]
    if arr.size == 0:
        return {f"{prefix}_{stat}": np.nan for stat in stats}
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr)),
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_range": float(np.max(arr) - np.min(arr)),
        f"{prefix}_q25": float(np.quantile(arr, 0.25)),
        f"{prefix}_median": float(np.quantile(arr, 0.50)),
        f"{prefix}_q75": float(np.quantile(arr, 0.75)),
        f"{prefix}_slope": float(arr[-1] - arr[0]) if arr.size >= 2 else 0.0,
    }


def compute_dynamic_summary(frame_df, col):
    arr = frame_df[col].to_numpy(dtype=np.float32)
    arr = arr[~np.isnan(arr)]
    if len(arr) < 2:
        return {
            f"{col}_velocity_mean_abs": np.nan,
            f"{col}_velocity_max_abs": np.nan,
            f"{col}_accel_mean_abs": np.nan,
            f"{col}_accel_max_abs": np.nan,
        }
    velocity = np.diff(arr)
    acceleration = np.diff(velocity)
    return {
        f"{col}_velocity_mean_abs": float(np.mean(np.abs(velocity))),
        f"{col}_velocity_max_abs": float(np.max(np.abs(velocity))),
        f"{col}_accel_mean_abs": float(np.mean(np.abs(acceleration))) if len(acceleration) else np.nan,
        f"{col}_accel_max_abs": float(np.max(np.abs(acceleration))) if len(acceleration) else np.nan,
    }


def compute_rule_based_measurements(frame_df):
    #Heuristic rule measurements for interpretation, not model inputs!
    out = {
        "rule_peak_torso_lean_deg": float(frame_df["torso_lean_deg"].max()),
        "rule_min_mean_knee_angle_deg": float(frame_df["mean_knee_angle_deg"].min()),
        "rule_max_knee_angle_asym_deg": float(frame_df["knee_angle_asym_deg"].max()),
        "rule_max_hip_angle_asym_deg": float(frame_df["hip_angle_asym_deg"].max()),
        "rule_peak_valgus_score_global": float(frame_df["valgus_score_global"].max()),
        "rule_peak_hip_to_ankle_x_norm_femur": float(frame_df["hip_to_ankle_x_norm_femur"].max()),
        "rule_peak_knee_to_ankle_x_norm_tibia": float(frame_df["knee_to_ankle_x_norm_tibia"].max()),
    }

    # intentional labeled as heuristics.
    # TODO: VALIDATE and cite before making stronger claims
    out["possible_excessive_torso_lean"] = int(out["rule_peak_torso_lean_deg"] > 45)
    out["possible_limited_depth"] = int(out["rule_min_mean_knee_angle_deg"] > 100)
    out["possible_knee_angle_asymmetry"] = int(out["rule_max_knee_angle_asym_deg"] > 15)
    out["possible_valgus_pattern"] = int(out["rule_peak_valgus_score_global"] > 0.20)
    return out


def compute_clip_level_features(points_seq):
    frame_df = pd.DataFrame([extract_frame_features(p) for p in points_seq])
    clip_features = {"num_pose_frames": len(frame_df)}

    summary_cols = [
        "mean_knee_angle_deg", "knee_angle_asym_deg", "mean_hip_angle_deg", "hip_angle_asym_deg",
        "torso_lean_deg", "femur_tibia_ratio", "torso_femur_ratio", "hip_to_ankle_x_norm_femur",
        "knee_to_ankle_x_norm_tibia", "valgus_score_global", "valgus_asymmetry", "stance_width_norm_torso",
    ]
    for col in summary_cols:
        if col in frame_df.columns:
            clip_features.update(summarize_series(frame_df[col], col))

    dynamic_cols = [
        "mean_knee_angle_deg", "mean_hip_angle_deg", "torso_lean_deg",
        "hip_to_ankle_x_norm_femur", "knee_to_ankle_x_norm_tibia", "valgus_score_global",
    ]
    for col in dynamic_cols:
        if col in frame_df.columns:
            clip_features.update(compute_dynamic_summary(frame_df, col))

    if "mean_knee_angle_deg" in frame_df.columns and len(frame_df) > 0:
        bottom_idx = int(frame_df["mean_knee_angle_deg"].idxmin())
        bottom = frame_df.loc[bottom_idx]
        for col in ["torso_lean_deg", "hip_to_ankle_x_norm_femur", "knee_to_ankle_x_norm_tibia", "valgus_score_global", "femur_tibia_ratio", "torso_femur_ratio"]:
            if col in bottom.index:
                clip_features[f"bottom_{col}"] = float(bottom[col])

    rule_measurements = compute_rule_based_measurements(frame_df)
    return clip_features, frame_df, rule_measurements

## 9. Build feature table

In [ ]:
def build_feature_table_from_registry(only_usable=True, save=True):
    registry_df = load_master_index().copy()
    if only_usable and "usable" in registry_df.columns:
        registry_df = registry_df[is_truthy_usable(registry_df["usable"])]

    feature_rows, rule_rows, failed_rows = [], [], []
    for _, row in registry_df.iterrows():
        clip_id = row["clip_id"]
        vector_path = CLIPS_DIR / clip_id / "pose_vectors.csv"
        if not vector_path.exists():
            failed_rows.append({"clip_id": clip_id, "error": "missing pose_vectors.csv"})
            continue
        try:
            points_seq, _ = load_vector_csv_as_points(vector_path)
            clip_features, _, rules = compute_clip_level_features(points_seq)
            feature_rows.append({"clip_id": clip_id, "vector_csv": str(vector_path), **clip_features})
            rule_rows.append({"clip_id": clip_id, **rules})
        except Exception as exc:
            failed_rows.append({"clip_id": clip_id, "error": str(exc)})

    features_df = pd.DataFrame(feature_rows)
    rules_df = pd.DataFrame(rule_rows)
    failed_df = pd.DataFrame(failed_rows)

    if save:
        features_df.to_csv(ARTIFACT_DIR / "clip_features.csv", index=False)
        rules_df.to_csv(ARTIFACT_DIR / "rule_measurements.csv", index=False)
        failed_df.to_csv(ARTIFACT_DIR / "feature_build_failures.csv", index=False)

    print("Feature table:", features_df.shape)
    print("Rule table:", rules_df.shape)
    print("Failures:", failed_df.shape)
    display(features_df.head())
    display(rules_df.head())
    if len(failed_df):
        display(failed_df.head(20))
    return features_df, rules_df, failed_df

# run after pose_vectors.csv files exist:
# features_df, rules_df, failed_df = build_feature_table_from_registry()

## 10. Merge labels and select model features

In [ ]:

features_path = ARTIFACT_DIR / "clip_features.csv"
if "features_df" not in globals():
    if not features_path.exists():
        raise FileNotFoundError("Run build_feature_table_from_registry() first, or provide clip_features.csv.")
    features_df = pd.read_csv(features_path)

master_df = load_master_index()
label_cols = ["clip_id", "good_form", "view_profile", "exercise_type", "load_type", "label_confidence", "view_confidence", "usable"]
labels_df = master_df[[c for c in label_cols if c in master_df.columns]].copy()

features_df = features_df.merge(labels_df, on="clip_id", how="left")
trainable_df = features_df.dropna(subset=["good_form"]).copy()
trainable_df["good_form"] = trainable_df["good_form"].astype(int)



metadata_cols = [
    c for c in [
        "clip_id", "source_csv", "vector_csv", "pose_csv", "url", "video_url", "start_time", "end_time",
        "good_form", "view_profile", "exercise_type", "load_type", "label_confidence", "view_confidence", "usable",
    ] if c in trainable_df.columns
]
explanation_cols = [c for c in trainable_df.columns if c.startswith("rule_") or c.startswith("possible_")]
excluded_cols = set(metadata_cols + explanation_cols)

feature_cols = [
    c for c in trainable_df.columns
    if c not in excluded_cols and pd.api.types.is_numeric_dtype(trainable_df[c])
]

if not feature_cols:
    raise ValueError("No numeric engineered feature columns found for modeling.")

X = trainable_df[feature_cols].copy()
y = trainable_df["good_form"].copy()

print("Trainable rows:", len(trainable_df))
print("Labels:")
print(y.value_counts())
print(f"Using {len(feature_cols)} numeric engineered feature columns.")
print("First 10 feature columns:", feature_cols[:10])

## 11. Baseline model training and repeated-split evaluation

In [ ]:
def make_models(random_state=42):
    return {
        "logistic_regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)),
        ]),
        "random_forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=random_state)),
        ]),
        "gradient_boosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(random_state=random_state)),
        ]),
    }


def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }


def repeated_split_evaluation(X, y, n_splits=20, test_size=0.25, base_seed=42):
    splitter = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=base_seed)
    rows = []

    for split_idx, (train_idx, test_idx) in enumerate(splitter.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        for model_name, model in make_models(random_state=base_seed + split_idx).items():
            fitted = clone(model)
            fitted.fit(X_train, y_train)
            y_pred = fitted.predict(X_test)
            rows.append({"split": split_idx, "model": model_name, **evaluate_predictions(y_test, y_pred)})

    return pd.DataFrame(rows)

results_df = repeated_split_evaluation(X, y, n_splits=20, test_size=0.25, base_seed=42)
summary_df = results_df.groupby("model").agg(["mean", "std", "min", "max"])

display(results_df.head())
display(summary_df)

results_df.to_csv(ARTIFACT_DIR / "model_comparison_repeated_splits.csv", index=False)

## 12. Fit final selected baseline and save artifacts

In [ ]:

mean_scores = results_df.groupby("model")["balanced_accuracy"].mean().sort_values(ascending=False)
best_model_name = mean_scores.index[0]
print("Selected model:", best_model_name)
print(mean_scores)

final_model = make_models(random_state=42)[best_model_name]
final_model.fit(X, y)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / f"{best_model_name}_squat_form_model.joblib"
schema_path = ARTIFACT_DIR / "feature_schema.json"

joblib.dump(final_model, model_path)
with open(schema_path, "w") as f:
    json.dump({
        "target": "good_form",
        "feature_columns": feature_cols,
        "excluded_metadata_columns": metadata_cols,
        "excluded_explanation_columns": explanation_cols,
        "selected_model": best_model_name,
        "note": "Rule-based measurements are interpretive heuristics and are excluded from model inputs.",
    }, f, indent=2)

print("Saved model:", model_path)
print("Saved schema:", schema_path)

## 13. Single holdout report for readability

Repeated splits above are better evidence. This single split is included only because it is easier to read and debug.

In [ ]:
stratify_y = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=stratify_y)

holdout_model = make_models(random_state=42)[best_model_name]
holdout_model.fit(X_train, y_train)
y_pred = holdout_model.predict(X_test)

print("Holdout metrics:")
print(evaluate_predictions(y_test, y_pred))
print("
Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("
Confusion matrix:")
print(confusion_matrix(y_test, y_pred))